In [ ]:
import sys, os
# Imports of vbn_utils, ccf_utils, decoding_utils, analysis_utils,
# notebook_utils, vbn_4day_utils resolve via vbn_code/utilities/.
_UTILS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'utilities'))
if _UTILS_DIR not in sys.path:
    sys.path.insert(0, _UTILS_DIR)

import h5py
import pandas as pd
import numpy as np
import os
from pathlib import Path
from matplotlib import pyplot as plt
import scipy.stats
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, Polygon
from matplotlib.collections import PolyCollection
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from functools import partial
import warnings
import vbn_utils
import decoding_utils as du
from analysis_utils import exponential_convolve
import ccf_utils
from vbn_utils import cumulative_hist, formatFigure, mean_sem_plot, make_iterable, get_unit_ids, bootstrap_ci
%matplotlib inline

## Data loading

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/ecephys_sessions.csv"

In [ ]:
units = pd.read_csv(unit_table_file)
units['cortical_layer'] = units['cortical_layer'].replace('3-Feb','2/3') # necessary since 2/3 sometimes gets incorrectly reformatted as a date

stim_table = pd.read_csv(stim_table_file)
stim_table = stim_table.drop(columns='Unnamed: 0')

active_tensor = h5py.File(active_tensor_file)

sessions_table = pd.read_csv(sessions_table_file)

In [ ]:
new_clusters = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/cluster_labels_122024.csv")
units = units.merge(new_clusters[['unit_id', 'cluster_labels_new_weak_cluster_12', 'used_in_clustering']], on='unit_id', how='left')
units.rename(columns={'cluster_labels_new_weak_cluster_12': 'cluster_labels_new', 'used_in_clustering': 'used_in_new_clustering'}, inplace=True)

In [ ]:
structure_tree = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/ccf_structure_tree_2017.csv")

In [ ]:
stim_table['is_shared'] = stim_table['image_name'].isin(['im083_r', 'im111_r'])

## Omission responses by cluster, cell type, and layer

In [ ]:
plt.rcParams['font.size'] = 16

### By cluster

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(8, 1)
fig.set_size_inches(7, 2.5*8)
for cluster in np.arange(1,9):
    ax = axes[cluster-1]
    session_list = list(active_tensor.keys())

    # unit_ids = units.loc[unit_filter]['unit_id'].values
    unit_ids = vbn_utils.get_unit_ids(units, 'VISall',cell_types='all', clusters=cluster, clustering='new')
                
    stim_filter = ['omitted']#, f'session_change_image_counts=={change_image_count}']

    omission_responses, sessionIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                *stim_filter, 
                                baseline_length=750, 
                                resp_window_length=750,
                                )

    all_omission_responses = []
    for oresps in omission_responses:
        for ors in oresps:
            all_omission_responses.append(exponential_convolve(ors, tau=3, symmetrical=True))


    time = np.arange(-750, 750, 1)
    # fig, ax = plt.subplots(figsize=(7, 3))

    ax.set_title(f'Cluster {cluster}', fontsize=16, pad=12, loc='right')
    vbn_utils.mean_sem_plot(np.array(all_omission_responses)*1000, ax, time, color='k')

    rectangle_height = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05

    filled_rect = Rectangle((-750, ax.get_ylim()[1]), 250, rectangle_height, color='gray', alpha=1, clip_on=False)
    ax.add_patch(filled_rect)

    open_rect = Rectangle((0, ax.get_ylim()[1]), 250, rectangle_height, edgecolor='gray', facecolor='none', linewidth=2, clip_on=False)
    ax.add_patch(open_rect)
    ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)
    loc = 'lower right' if cluster == 5 else 'upper right'
    borderpad = 1.5 if cluster == 5 else 0.5
    inset_ax = inset_axes(ax, width="20%", height="40%", loc=loc, borderpad=borderpad)  # Relative to the main plot


    vbn_utils.mean_sem_plot(np.array(all_omission_responses)[:,600:1100]*1000, inset_ax, time[600:1100], color='k')
    # inset_ax.plot(time[-250:], np.array(all_omission_responses).mean(axis=0)[-250:]*1000, color='k')
    inset_ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)
    vbn_utils.formatFigure(fig, inset_ax)


    ax.set_xlim(-750, 750)
    vbn_utils.formatFigure(fig, ax)
plt.tight_layout()

### By cell type

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(4, 1)
fig.set_size_inches(7, 2.5*4)
for icell, cell_type in enumerate(['RS', 'FS', 'SST', 'VIP']):
    ax = axes[icell]
    session_list = list(active_tensor.keys())

    # unit_ids = units.loc[unit_filter]['unit_id'].values
    unit_ids = vbn_utils.get_unit_ids(units, 'VISall',cell_types=cell_type, clusters='all', clustering='new', experience='all')
                
    stim_filter = ['omitted']#, f'session_change_image_counts=={change_image_count}']

    omission_responses, sessionIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                *stim_filter, 
                                baseline_length=750, 
                                resp_window_length=750,
                                )

    all_omission_responses = []
    for oresps in omission_responses:
        for ors in oresps:
            all_omission_responses.append(exponential_convolve(ors, tau=3, symmetrical=True))


    time = np.arange(-750, 750, 1)
    # fig, ax = plt.subplots(figsize=(7, 3))

    ax.set_title(f'Cell Type {cell_type}', fontsize=16, pad=12, loc='right')
    vbn_utils.mean_sem_plot(np.array(all_omission_responses)*1000, ax, time, color='k')

    if cell_type == 'VIP':
        ax.set_ylim(-3, 15)

    rectangle_height = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05

    filled_rect = Rectangle((-750, ax.get_ylim()[1]), 250, rectangle_height, color='gray', alpha=1, clip_on=False)
    ax.add_patch(filled_rect)

    open_rect = Rectangle((0, ax.get_ylim()[1]), 250, rectangle_height, edgecolor='gray', facecolor='none', linewidth=2, clip_on=False)
    ax.add_patch(open_rect)
    ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)

    # ax.text(125, ax.get_ylim()[1]+0.02, 'Pre-omission image', ha='center', va='bottom', fontsize=12, color='w')
    # ax.text(875, ax.get_ylim()[1]+0.02, 'Omission', ha='center', va='bottom', fontsize=12)
    ax.set_xlim(-750, 750)

    vbn_utils.formatFigure(fig, ax)

    loc = 'lower right' if cell_type == 'VIP' else 'upper right'
    borderpad = 1.5 if cell_type == 'VIP' else 0.5
    inset_ax = inset_axes(ax, width="20%", height="40%", loc=loc, borderpad=borderpad)  # Relative to the main plot

    vbn_utils.mean_sem_plot(np.array(all_omission_responses)[:,600:1100]*1000, inset_ax, time[600:1100], color='k')
    # inset_ax.plot(time[-250:], np.array(all_omission_responses).mean(axis=0)[-250:]*1000, color='k')
    inset_ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)
    vbn_utils.formatFigure(fig, inset_ax)

plt.tight_layout()

### By cortical layer

In [ ]:
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(4, 1)
fig.set_size_inches(7, 2.5*4)
for ilayer, layer in enumerate(['2/3', '4', '5', '6']):
    ax = axes[ilayer]
    session_list = list(active_tensor.keys())

    # unit_ids = units.loc[unit_filter]['unit_id'].values
    unit_ids = vbn_utils.get_unit_ids(units, 'VISall',cell_types='all', layers=layer, clusters='all', clustering='new')
                
    stim_filter = ['omitted']#, f'session_change_image_counts=={change_image_count}']

    omission_responses, sessionIds = vbn_utils.unit_averaged_psth(active_tensor_file, 
                                stim_table, 
                                session_list, 
                                unit_ids, 
                                *stim_filter, 
                                baseline_length=750, 
                                resp_window_length=750,
                                )

    all_omission_responses = []
    for oresps in omission_responses:
        for ors in oresps:
            all_omission_responses.append(exponential_convolve(ors, tau=3, symmetrical=True))


    time = np.arange(-750, 750, 1)
    # fig, ax = plt.subplots(figsize=(7, 3))

    ax.set_title(f'Layer {layer} RS', fontsize=16, pad=12, loc='right')
    vbn_utils.mean_sem_plot(np.array(all_omission_responses)*1000, ax, time, color='k')

    rectangle_height = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05

    filled_rect = Rectangle((-750, ax.get_ylim()[1]), 250, rectangle_height, color='gray', alpha=1, clip_on=False)
    ax.add_patch(filled_rect)

    open_rect = Rectangle((0, ax.get_ylim()[1]), 250, rectangle_height, edgecolor='gray', facecolor='none', linewidth=2, clip_on=False)
    ax.add_patch(open_rect)

    ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)
    # ax.text(125, ax.get_ylim()[1]+0.02, 'Pre-omission image', ha='center', va='bottom', fontsize=12, color='w')
    # ax.text(875, ax.get_ylim()[1]+0.02, 'Omission', ha='center', va='bottom', fontsize=12)
    ax.set_xlim(-750, 750)
    vbn_utils.formatFigure(fig, ax)

    loc =  'upper right'
    borderpad = 0.5
    inset_ax = inset_axes(ax, width="20%", height="40%", loc=loc, borderpad=borderpad)  # Relative to the main plot

    vbn_utils.mean_sem_plot(np.array(all_omission_responses)[:,600:1100]*1000, inset_ax, time[600:1100], color='k')
    # inset_ax.plot(time[-250:], np.array(all_omission_responses).mean(axis=0)[-250:]*1000, color='k')
    inset_ax.axhline(np.array(all_omission_responses).mean(axis=0)[0]*1000, color='k', linestyle='--', linewidth=1)
    vbn_utils.formatFigure(fig, inset_ax)

plt.tight_layout()